In [1]:
## This notebook is meant to document (a) how I downloaded the appropriate data from ONE (b) how I formatted it (and what each item represents)


In [1]:
import fastplotlib as fpl
import masknmf #Realistically do not need this import here
import numpy as np
import os
import torch
from typing import *

from one.api import ONE
one = ONE()
assert not one.offline, 'ONE must be connect to Alyx for searching imaging sessions'


import pathlib
from pathlib import Path
%load_ext autoreload

No windowing system present. Using surfaceless platform
No config found!
No config found!
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),NVIDIA TITAN RTX,DiscreteGPU,Vulkan,555.42.02
❗ limited,"llvmpipe (LLVM 12.0.0, 256 bits)",CPU,Vulkan,Mesa 21.2.6 (LLVM 12.0.0)
❌,NVIDIA TITAN RTX/PCIe/SSE2,Unknown,OpenGL,3.3.0 NVIDIA 555.42.02


Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


In [2]:
## This is the EID we will use here

eids = ONE().search(
    subject='SP044', 
    number="001",
    date_range=['2023-06-26', '2023-06-28'],
)
eid_used = eids[0]

In [6]:
one.list_datasets(eid_used)

['_ibl_experiment.description.yaml',
 'alf/FOV_00/_suite2p_ROIData.raw.zip',
 'alf/FOV_00/mpci.ROIActivityDeconvolved.npy',
 'alf/FOV_00/mpci.ROIActivityF.npy',
 'alf/FOV_00/mpci.ROINeuropilActivityF.npy',
 'alf/FOV_00/mpci.badFrames.npy',
 'alf/FOV_00/mpci.mpciFrameQC.npy',
 'alf/FOV_00/mpci.times.npy',
 'alf/FOV_00/mpciFrameQC.names.tsv',
 'alf/FOV_00/mpciMeanImage.brainLocationIds_ccf_2017_estimate.npy',
 'alf/FOV_00/mpciMeanImage.images.npy',
 'alf/FOV_00/mpciMeanImage.mlapdv_estimate.npy',
 'alf/FOV_00/mpciROITypes.names.tsv',
 'alf/FOV_00/mpciROIs.brainLocationIds_ccf_2017_estimate.npy',
 'alf/FOV_00/mpciROIs.cellClassifier.npy',
 'alf/FOV_00/mpciROIs.masks.sparse_npz',
 'alf/FOV_00/mpciROIs.mlapdv_estimate.npy',
 'alf/FOV_00/mpciROIs.mpciROITypes.npy',
 'alf/FOV_00/mpciROIs.neuropilMasks.sparse_npz',
 'alf/FOV_00/mpciROIs.stackPos.npy',
 'alf/FOV_00/mpciROIs.uuids.csv',
 'alf/FOV_00/mpciStack.timeshift.npy',
 'alf/FOV_01/_suite2p_ROIData.raw.zip',
 'alf/FOV_01/mpci.ROIActivityDe

# Download the left + right camera views

In [3]:
raw_vid = one.load_object(eid_used, 'leftCamera', collection='raw_video_data', download_only=True)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64.0 [00:37<00:00,  1.69it/s]


In [7]:
raw_vid = one.load_object(eid_used, 'rightCamera', collection='raw_video_data', download_only=True)

(S3) /media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/raw_video_data/_iblrig_rightCamera.frameData.bin: 100%|████████████████████████████████| 20.6M/20.6M [00:00<00:00, 22.9MB/s]
(S3) /media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/raw_video_data/_iblrig_rightCamera.raw.mp4: 100%|███████████████████████████████████████| 2.61G/2.61G [00:22<00:00, 114MB/s]


In [29]:
raw_vid

[PosixALFPath('/media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/raw_video_data/_iblrig_rightCamera.frameData.bin'),
 PosixALFPath('/media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/raw_video_data/_iblrig_rightCamera.raw.mp4')]

In [39]:
## These are the frame acquisition times!
right_camera_times = one.load_dataset(eid_used, f'*rightCamera.times*', collection= 'alf', download_only = True)
left_camera_times = one.load_dataset(eid_used, f'*leftCamera.times*', collection='alf', download_only = True)

In [40]:
right_camera_times

PosixALFPath('/media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/alf/_ibl_rightCamera.times.npy')

In [28]:
right_camera_times

array([  14.6005,   14.607 ,   14.6135, ..., 4300.9825, 4300.989 ,
       4300.9955], shape=(644673,))

In [36]:
1 / (right_camera_times[1] - right_camera_times[0])#Frame rate

np.float64(153.846153846134)

In [35]:
left_camera_times

array([   7.453 ,    7.48  ,    7.4865, ..., 4293.8605, 4293.8675,
       4293.874 ], shape=(639919,))

In [37]:
1 / (left_camera_times[1] - left_camera_times[0]) #Frame rate

np.float64(37.03703703703685)

In [34]:
left_camera_times

array([   7.453 ,    7.48  ,    7.4865, ..., 4293.8605, 4293.8675,
       4293.874 ], shape=(639919,))

# Download the relevant keypoint data for both of these

In [8]:
behavior_features_leftcam = one.load_object(eid_used, 'leftCamera', collection='alf') #This should be a dict (or an Alf bunch object) 

In [9]:
behavior_features_rightcam = one.load_object(eid_used, 'rightCamera', collection='alf') #This should be a dict (or an Alf bunch object) 

ERROR! Session/line number was not unique in database. History logging moved to new session 10254



(S3) /media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/alf/_ibl_rightCamera.features.pqt:   0%|                                                         | 0.00/137k [00:00<?, ?B/s]
(S3) /media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/alf/_ibl_rightCamera.features.pqt: 100%|█████████████████████████████████████████████████| 137k/137k [00:00<00:00, 1.01MB/s]


In [19]:
print(list(behavior_features_rightcam.keys()))

['times', 'dlc', 'ROIMotionEnergy', 'features']


In [16]:
behavior_features_rightcam['times']

array([  14.6005,   14.607 ,   14.6135, ..., 4300.9825, 4300.989 ,
       4300.9955], shape=(644673,))

In [25]:
np.savez("behavior_features_rightcam.npz", data = behavior_features_rightcam)
np.savez("behavior_features_leftcam.npz", data = behavior_features_leftcam)

In [30]:
bhfr_2 = np.load("behavior_features_rightcam.npz", allow_pickle = True)['data'].item()

In [33]:
bhfr_2['dlc']#['nose_tip_x']

,nose_tip_x,nose_tip_y,nose_tip_likelihood,pupil_top_r_x,pupil_top_r_y,pupil_top_r_likelihood,pupil_right_r_x,pupil_right_r_y,pupil_right_r_likelihood,pupil_bottom_r_x,...,tube_top_likelihood,tube_bottom_x,tube_bottom_y,tube_bottom_likelihood,tongue_end_l_x,tongue_end_l_y,tongue_end_l_likelihood,tongue_end_r_x,tongue_end_r_y,tongue_end_r_likelihood
0,600.220875,211.430708,1.000000,230.739780,323.812790,0.685135,229.461790,324.473894,0.664902,233.980808,...,0.999811,536.642269,285.508163,0.999201,501.826347,262.951015,0.000126,579.395747,229.303270,0.000035
1,600.103638,211.430708,1.000000,230.737291,323.813700,0.684990,229.459129,324.474096,0.664569,233.980808,...,0.999819,536.583775,285.508163,0.999095,501.808098,263.005577,0.000452,579.339150,229.297662,0.000024
2,600.097067,211.430708,1.000000,230.735729,323.814384,0.684939,229.457560,324.474449,0.664480,233.980783,...,0.999815,536.583775,285.509476,0.999099,501.774414,265.253811,0.000452,579.339150,229.235634,0.000027
3,600.097067,210.886768,1.000000,230.732933,323.816061,0.684884,229.455654,324.474449,0.664252,233.980604,...,0.999818,536.583775,285.508163,0.999092,500.370049,265.171383,0.000596,579.283039,229.235634,0.000022
4,600.097067,210.626431,1.000000,230.732933,323.820393,0.685168,229.452883,324.476854,0.664141,233.980604,...,0.999810,536.583775,285.503227,0.999088,500.370049,265.253811,0.000598,579.238566,229.303270,0.000016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644668,596.925274,206.904957,1.000000,230.406757,323.588314,0.549645,229.394573,324.059752,0.535340,233.708521,...,0.999813,536.021355,287.194714,0.996624,547.077503,276.199940,0.000156,535.211834,251.779840,0.000080
644669,597.011606,207.627151,0.089772,230.406757,323.588564,0.549617,229.394531,324.059805,0.535375,233.708242,...,0.999825,536.021355,287.172558,0.995652,547.043980,276.171169,0.000158,535.207428,251.756205,0.000176
644670,597.589916,207.627151,1.000000,230.406588,323.588818,0.549709,229.394182,324.059908,0.535472,233.707115,...,0.999835,536.021355,287.116264,0.996356,547.043980,276.116634,0.000067,535.211834,251.756205,0.000058
644671,597.669973,207.627151,1.000000,230.406588,323.588818,0.549866,229.394182,324.059908,0.535677,233.707115,...,0.999640,536.041317,287.046555,0.995685,547.043980,276.108150,0.000251,538.122643,251.630941,0.000077


In [12]:
## See the public documentation here explaining what the features are in the DLC outputs: 
## https://docs.google.com/document/u/1/d/e/2PACX-1vS2777bCbDmMre-wyeDr4t0jC-0YsV_uLtYkfS3h9zTwgC7qeMk-GUqxPqcY7ylH17I1Vo1nIuuj26L/pub
dlc = behavior_features_rightcam['dlc']

In [15]:
print(list(behavior_features_rightcam.keys()))

['times', 'dlc', 'ROIMotionEnergy', 'features']


# Download raw data

In [62]:
# one.load_dataset(eid_used, '*', collection = 'suite2p/plane0/', download_only = True)
# one.load_dataset(eid_used, '*_suite2p_ROIData*', collection = 'alf/FOV_00/', download_only = True)

motion_bin_data = one.load_dataset(eid_used, 
                        'imaging.frames_motionRegistered.bin', 
                        collection='suite2p/plane7', download_only=True)

file = one.load_dataset(eid_used, 
                        '_suite2p_ROIData.raw.zip',
                        collection='alf/FOV_07',
                        download_only=True)

times_temp = one.load_dataset(eid_used,
                              'mpci.times',
                              collection='alf/FOV_07', 
                              download_only = True)

(S3) /media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/suite2p/plane7/imaging.frames_motionRegistered.bin: 100%|███████████████████████████████| 7.12G/7.12G [01:01<00:00, 117MB/s]
(S3) /media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/alf/FOV_07/_suite2p_ROIData.raw.zip: 100%|████████████████████████████████████████████████| 337M/337M [00:03<00:00, 103MB/s]
(S3) /media/app2139/SanDisk_1/IBL_Alyx/cortexlab/Subjects/SP044/2023-06-27/001/alf/FOV_07/mpci.times.npy: 100%|█████████████████████████████████████████████████████████| 217k/217k [00:00<00:00, 1.56MB/s]


# Download stim time data

In [63]:
trials = one.load_object(eid_used, 'trials')

In [67]:
np.savez("Trials_ALF.npz", trials = trials)